# 응용 모의고사 Set 4 — 정답 — 고객 집계와 군집화

- 데이터: `sales_pos.csv`
- 난이도: 기존 Set 01~06과 유사
- 구성: **공통 전처리 → Q1 통계 → Q2 상관분석 → Q3 모델링**
- 모든 문항은 공통 전처리 결과를 이어서 사용합니다.
- 전처리 완료 후 데이터는 **5,891행**이어야 합니다. 행 수가 다르면 다음 문제로 넘어가기 전에 전처리를 확인하세요.

정답 노트북은 `../answers/`에 있습니다.

## 공통 전처리 정답

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../dataset/sales_pos.csv')
work = df.copy()
category_cols = ['prod_cat1', 'prod_cat2', 'prod_cat3']
work[category_cols] = work[category_cols].fillna(0).astype(int).astype(str)
work['prod_cat'] = work[category_cols].agg('-'.join, axis=1)
base = work.groupby('user').agg(
    gender=('gender', 'first'),
    age=('age_group', 'first'),
    job=('job', 'first'),
    city=('city', 'first'),
    marital=('marital', 'first'),
    prod_count=('prod', 'nunique'),
    category_count=('prod_cat', 'nunique'),
    total_purchase=('purchase', 'sum'),
    transaction_count=('purchase', 'size'),
    avg_purchase=('purchase', 'mean')
).reset_index()
assert len(base) == 5891
display(base.head())

## Q1 정답

In [ ]:
city_mean = base.groupby('city')['category_count'].mean()
answer_q1 = round(city_mean.max() - city_mean.min(), 2)
display(city_mean, answer_q1)  # 22.48

## Q2 정답

In [ ]:
features = ['prod_count', 'category_count', 'avg_purchase']
purchase_corr = base[['total_purchase'] + features].corr()['total_purchase'].drop('total_purchase')
answer_var_q2 = purchase_corr.abs().idxmax()
answer_coef_q2 = round(purchase_corr.loc[answer_var_q2], 3)
display(answer_var_q2, answer_coef_q2)  # prod_count, 0.979

## Q3 정답

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

model_df = base.drop(columns='user').copy()
model_df['gender'] = model_df['gender'].replace({'M': 1, 'F': 0})
model_df['age'] = pd.to_numeric(
    model_df['age'].str.extract(r'(\d+)', expand=False), errors='coerce'
)
X = pd.get_dummies(model_df, columns=['job', 'city'])
X_scaled = MinMaxScaler().fit_transform(X)
scores = {}
for k in [3, 4, 5, 6]:
    labels = KMeans(n_clusters=k, random_state=321, n_init=10).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
answer_k_q3 = pd.Series(scores).idxmax()
answer_score_q3 = round(scores[answer_k_q3], 3)
display(pd.Series(scores), answer_k_q3, answer_score_q3)  # 3, 0.238